# Notebook 02 — Pearson Correlation & Time Series
**Part A — Temporal analysis**

## What this notebook does
1. Computes Pearson correlations between all 7 meteorological variables and 4 bioaerosol targets  
2. Produces the **correlation heatmap** with significance stars (Fig 2 in paper)  
3. Produces the **daily time series** plot across all 6 stations (Fig 1 in paper)  
4. Prints the full numeric correlation table

## Inputs
- `df_analysis.csv` — output from Notebook 01

### Action Required: Generate `df_analysis.csv`

The `df_analysis.csv` file, which is an input for this notebook, is generated by `01_data_preparation.ipynb`. Please run the `01_data_preparation.ipynb` notebook first to create this file. Once it has finished executing, you can return to this notebook and proceed.

In [15]:
# ===================================================================
# COLAB SETUP — run this cell first if using Google Colab
# ===================================================================
import os, sys

# Clean up any previous clone to avoid nesting issues
!rm -rf geoai-bioaerosol-prediction

# Clone the GitHub repo
!git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction.git

# Change into the repository directory
os.chdir('/content') # Go to Colab's default root first
os.chdir('geoai-bioaerosol-prediction') # Then change to the cloned repo directory

# Option B: Mount Google Drive and navigate to your folder
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/geoai-bioaerosol-prediction')

# Install dependencies
# !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn -q

print('Current directory:', os.getcwd())
print('Python:', sys.version[:10])

# List all files to verify contents and locate df_analysis.csv
print('\nRepository contents:')
!ls -R

Cloning into 'geoai-bioaerosol-prediction'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 156 (delta 44), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 4.63 MiB | 18.37 MiB/s, done.
Resolving deltas: 100% (44/44), done.
Current directory: /content/geoai-bioaerosol-prediction
Python: 3.12.13 (m

Repository contents:
.:
 01_data_preparation.ipynb			 grassland1.dbf
 02_pearson_correlation_timeseries.ipynb	 grassland1.prj
 03_random_forest_permutation_importance.ipynb	 grassland1.sbn
 04_shap_analysis.ipynb				 grassland1.sbx
 05_lulc_spatial_analysis.ipynb			 grassland1.shp
 06_height_gradient_analysis.ipynb		 grassland1.shx
 alongroadsfinal.cpg				 grasslands
 alongroadsfinal.dbf				 insidegrasstotal.cpg
 alongroadsfinal.prj				 insidegrasstotal.dbf
 alongroadsfinal.sbn				 insidegrasstotal.prj
 alongroadsfinal.sbx				 insidegrasstotal.sbn
 alongroa

In [17]:
# ── Install dependencies (uncomment on Colab) ─────────────────────
# !pip install matplotlib seaborn scipy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('df_analysis.csv')
for col in ['Aspergillus_conc', 'Alternaria_conc']:
    df[col] = df[col].fillna(df[col].median())

# ── Define variables ─────────────────────────────────────────────
MET_FEATURES = ['GHI', 'Tamb', 'RH', 'WS', 'BP', 'sin_WD', 'cos_WD']
FEAT_LABELS  = ['GHI', 'Temp', 'RH', 'Wind Speed', 'Pressure', 'sin(WD)', 'cos(WD)']

TARGETS = {
    'pollen_conc':      'Pollen (grain/m³)',
    'fungus_conc':      'Total Fungus (grain/m³)',
    'Aspergillus_conc': 'Aspergillus (grain/m³)',
    'Alternaria_conc':  'Alternaria (grain/m³)',
}
COLORS = {
    'pollen_conc':      '#378ADD',
    'fungus_conc':      '#D85A30',
    'Aspergillus_conc': '#1D9E75',
    'Alternaria_conc':  '#BA7517',
}

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Stations: {sorted(df["station"].unique())}')
print(f'Dates:    {df["date"].nunique()} unique dates')

FileNotFoundError: [Errno 2] No such file or directory: 'df_analysis.csv'

### Running `01_data_preparation.ipynb`

To generate `df_analysis.csv`, you can run `01_data_preparation.ipynb` directly from this notebook using the `%run` magic command. This will execute all the cells in that notebook, and upon completion, `df_analysis.csv` should be created in the current working directory.

In [18]:
# Run the 01_data_preparation.ipynb notebook to generate df_analysis.csv
%run 01_data_preparation.ipynb

Mounted at /content/drive
fatal: destination path 'geoai-bioaerosol-prediction' already exists and is not an empty directory.
Current directory: /content/geoai-bioaerosol-prediction/geoai-bioaerosol-prediction
Python: 3.12.13 (m


AttributeError: module 'osgeo.gdal' has no attribute 'config'

AttributeError: module 'osgeo.gdal' has no attribute 'config'

After running the above cell, `df_analysis.csv` should now be available. You can re-run the `pd.read_csv('df_analysis.csv')` cell (cell `xQs-PT9HNvB_`) to load the data.

## Figure 1 — Daily time series across 6 stations
Shows day-to-day variation in pollen and fungal concentrations at each site.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
palette = plt.cm.tab10(np.linspace(0, 0.6, 6))

for sta_i, sta in enumerate(sorted(df['station'].unique())):
    sub = df[df['station'] == sta].sort_values('date')
    axes[0].plot(sub['date'], sub['pollen_conc'],
                 marker='o', ms=5, label=f'Station {sta}',
                 color=palette[sta_i], lw=1.5)
    axes[1].plot(sub['date'], sub['fungus_conc'],
                 marker='s', ms=5, color=palette[sta_i], lw=1.5)

for ax, ylabel in zip(axes, ['Pollen concentration (grain/m³)',
                               'Total Fungus concentration (grain/m³)']):
    ax.set_ylabel(ylabel, fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='x', rotation=35, labelsize=8)
    ax.grid(axis='y', alpha=0.25, linestyle='--')

axes[0].legend(ncol=3, fontsize=9, loc='upper right', framealpha=0.8)
fig.suptitle('Daily bioaerosol concentrations across 6 sampling stations\nNUST H-12 campus, April 2016',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_timeseries.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_timeseries.png')

## Figure 2 — Pearson correlation heatmap
### What Pearson correlation measures
Pearson r measures the **linear relationship** between two variables:  
- r = +1 → perfect positive linear relationship  
- r = 0  → no linear relationship  
- r = -1 → perfect negative linear relationship  

p-values test whether the correlation is significantly different from 0.

### Why we use it here
To identify which meteorological variables are most linearly associated  
with daily bioaerosol concentrations at each site.

### Important caveat
All 6 stations share a **single weather station** record.  
So correlations reflect **day-to-day temporal variation** only —  
not spatial differences between sites.

In [ ]:
# ── Compute Pearson r and p-values for all feature × target pairs ──
corr_data = {}
pval_data = {}

for tcol in TARGETS:
    corr_data[tcol] = []
    pval_data[tcol] = []
    for feat in MET_FEATURES:
        # dropna to handle any missing values
        mask = df[[tcol, feat]].dropna()
        r, p = stats.pearsonr(mask[feat], mask[tcol])
        corr_data[tcol].append(round(r, 3))
        pval_data[tcol].append(p)

corr_df = pd.DataFrame(corr_data, index=FEAT_LABELS).T
pval_df  = pd.DataFrame(pval_data, index=FEAT_LABELS).T

# ── Print numeric table ─────────────────────────────────────────
print('Pearson correlation table (r values):')
print(corr_df.round(3).to_string())
print()
print('p-value table:')
print(pval_df.round(4).to_string())

In [ ]:
# ── Build heatmap with significance annotations ─────────────────
fig, ax = plt.subplots(figsize=(11, 4.5))

sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'},
            ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.85})

# Overlay significance stars below each cell value
for i, tcol in enumerate(TARGETS):
    for j, feat in enumerate(FEAT_LABELS):
        p   = pval_df.loc[tcol, feat]
        sym = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        if sym:
            ax.text(j + 0.5, i + 0.82, sym, ha='center', va='center',
                    fontsize=8, color='black')

ax.set_yticklabels([TARGETS[t] for t in TARGETS], rotation=0, fontsize=10)
ax.set_xticklabels(FEAT_LABELS, rotation=30, ha='right', fontsize=10)
ax.set_title('Pearson correlations: meteorological variables vs bioaerosol concentrations\n'
             '(* p<0.05   ** p<0.01   *** p<0.001)',
             fontsize=11, pad=12)
plt.tight_layout()
plt.savefig('fig_pearson_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_pearson_heatmap.png')

## Per-site Pearson correlations
The heatmap above pools all stations. Now we compute per-site correlations  
as shown in Table 4 of the paper — this shows that Site 2 (open space)  
has the only significant wind speed correlations, while other sites show  
weak or inconsistent relationships.

In [ ]:
print('Per-site Pearson correlations (r) — matching Table 4 in paper')
print('P = Pollen | F = Total Fungus')
print()

key_vars = ['GHI', 'Tamb', 'RH', 'WS']

for sta in range(1, 7):
    sub = df[df['station'] == sta]
    print(f'Station {sta}:')
    for feat in key_vars:
        for tcol, label in [('pollen_conc','P'), ('fungus_conc','F')]:
            pair = sub[[feat, tcol]].dropna()
            if len(pair) >= 4:
                r, p = stats.pearsonr(pair[feat], pair[tcol])
                sig  = '*' if p < 0.05 else ''
                print(f'  {label}:{feat:12s}  r={r:+.3f}  p={p:.3f} {sig}')
    print()